In [7]:
import sys
sys.path.insert(0, '/Users/kevinlaventure/python_code')

import pandas as pd
import numpy as np
from python_module.pricing_model import SABRModel

# Define parameters
F = 100  # Forward price
r = 0.05  # Risk-free rate

# SABR model parameters
alpha = 0.2  # Volatility of volatility
beta = 1  # CEV exponent (0.8 is typical for rates)
rho = -0.9  # Correlation between forward and volatility
nu = 0.9  # Vol of vol



# Create maturity range (1 day to 60 days) in years
maturities_days = np.arange(1, 61)
maturities_years = maturities_days / 365

# Create strike range (50 to 200)
strikes = np.arange(50, 201, 1)

# Slide scenarios (-1%, -5%, -10%)
slides = [-0.01, -0.05, -0.10, -0.3, ]

# Create dataframe
results = []

for maturity_years in maturities_years:
    for strike in strikes:
        option_type = 'put' if strike < F else 'call'
        row = {
            'maturity_days': int(maturity_years * 365),
            'maturity_years': maturity_years,
            'strike': strike,
        }
        
        # Compute option price with slide scenarios using SABR model
        option_result = SABRModel.compute_option(
            F=F, 
            K=strike, 
            T=maturity_years, 
            r=r, 
            alpha=alpha,
            beta=beta,
            rho=rho,
            nu=nu,
            option_type=option_type,
            slide_scenario=slides,
            slide_type='spot_vol',
            slide_compute='option_pnl'
        )
        
        # Add implied volatility and base option price and greeks
        row['IV'] = option_result.get('IV', np.nan)
        row['price'] = option_result.get('price', np.nan)
        row['delta'] = option_result.get('delta', np.nan)
        row['gamma'] = option_result.get('gamma', np.nan)
        row['vega'] = option_result.get('vega', np.nan)
        row['theta'] = option_result.get('theta', np.nan)
        
        # Add slide PnL columns
        for slide in slides:
            slide_pct = f"slide_{int(slide*100)}pct"
            row[slide_pct] = option_result.get(slide, np.nan)
        
        results.append(row)

df = pd.DataFrame(results)

In [13]:
df['ask_price'] = df['price'] + 0.1

,maturity_days,maturity_years,strike,IV,price,delta,gamma,vega,theta,slide_-1pct,slide_-5pct,slide_-10pct
0,1,0.002740,50,0.431565,2.345372e-208,0.000000e+00,4.341524e-206,5.133287e-207,-1.604368e-205,1.017355e-196,9.908535e-157,5.958446e-118
1,1,0.002740,51,0.425802,4.358502e-202,0.000000e+00,8.035076e-200,9.373560e-201,-2.890508e-199,1.275440e-190,2.931354e-151,3.629997e-113
2,1,0.002740,52,0.420119,6.716105e-196,0.000000e+00,1.232361e-193,1.418462e-194,-4.315713e-193,1.324035e-184,7.140411e-146,1.807026e-108
3,1,0.002740,53,0.414514,8.619294e-190,0.000000e+00,1.573242e-187,1.786661e-188,-5.363446e-187,1.142948e-178,1.437122e-140,7.369848e-104
4,1,0.002740,54,0.408984,9.249953e-184,0.000000e+00,1.678383e-181,1.880634e-182,-5.570221e-181,8.235625e-173,2.397266e-135,2.468047e-99
...,...,...,...,...,...,...,...,...,...,...,...,...
9055,60,0.164384,196,0.159563,1.028885e-25,1.689262e-25,2.732440e-25,7.167058e-25,-1.380129e-25,1.781907e-25,9.008712e-24,2.683733e-22
9056,60,0.164384,197,0.160323,7.620280e-26,1.248559e-25,2.015510e-25,5.311754e-25,-1.027732e-25,1.328412e-25,6.807444e-24,2.064268e-22
9057,60,0.164384,198,0.161079,5.665318e-26,9.263362e-26,1.492331e-25,3.951496e-25,-7.681519e-26,9.939666e-26,5.161723e-24,1.592739e-22
9058,60,0.164384,199,0.161831,4.227555e-26,6.898248e-26,1.109058e-25,2.950362e-25,-5.762164e-26,7.463908e-26,3.927011e-24,1.232670e-22


In [14]:
df.loc[df['strike']==100]

,maturity_days,maturity_years,strike,IV,price,delta,gamma,vega,theta,slide_-1pct,slide_-5pct,slide_-10pct
50,1,0.002740,100,0.199970,0.417510,0.502019,0.381089,0.020878,-0.302278,-0.318670,-0.417509,-0.417510
201,2,0.005479,100,0.199940,0.590275,0.502814,0.269471,0.029522,-0.213620,-0.362972,-0.589935,-0.590275
352,2,0.008219,100,0.199910,0.722726,0.503408,0.220022,0.036152,-0.174319,-0.383183,-0.719702,-0.722725
503,4,0.010959,100,0.199879,0.834288,0.503898,0.190545,0.041738,-0.150878,-0.395074,-0.824389,-0.834282
654,5,0.013699,100,0.199849,0.932489,0.504320,0.170428,0.046658,-0.134872,-0.402974,-0.911213,-0.932444
805,5,0.016438,100,0.199819,1.021192,0.504695,0.155579,0.051103,-0.123050,-0.408610,-0.984529,-1.021000
956,7,0.019178,100,0.199789,1.102691,0.505034,0.144038,0.055189,-0.113856,-0.412823,-1.047325,-1.102136
1107,8,0.021918,100,0.199759,1.178482,0.505345,0.134736,0.058991,-0.106441,-0.416077,-1.101760,-1.177226
1258,9,0.024658,100,0.199729,1.249603,0.505632,0.127030,0.062560,-0.100296,-0.418651,-1.149435,-1.247195
1409,10,0.027397,100,0.199699,1.316812,0.505900,0.120511,0.065934,-0.095094,-0.420722,-1.191566,-1.312708
